# Klasifikimi i Imazheve HAM10000 me Decision Tree (Pemë Vendimesh)

Ky notebook përdor datasetin e filtruar HAM10000 për të trajnuar një model klasifikimi binar (`mel` vs `nv`) duke përdorur algoritmin **Decision Tree** (Pemë Vendimesh).

Modeli merr si input dy lloje karakteristikash:
1. **Imazhet**: të cilat ndryshohen në përmasa `32x32` piksela, kthehen në vektorë të sheshtë (flattened) dhe normalizohen.
2. **Metadata**: duke përfshirë moshën, gjininë dhe lokalizimin e lesionit (të koduara me One-Hot Encoding).

### 1. Importimi i Bibliotekave të Nevojshme

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import joblib

### 2. Përcaktimi i Folderave (Paths)

In [ ]:
BASE_DIR = os.getcwd()

METADATA_PATH = os.path.join(
    BASE_DIR,
    "HAM10000_metadata_filtered.csv"
)

MEL_DIR = os.path.join(
    BASE_DIR,
    "mel"
)

NV_DIR = os.path.join(
    BASE_DIR,
    "nv"
)

print("Base directory:", BASE_DIR)

### 3. Ngarkimi dhe Pastrimi i Metadata
Mbushim moshat që mungojnë dhe kodojmë kolonat kategoriale (`sex` dhe `localization`) në vlera numerike.

In [ ]:
# Ngarkojmë CSV
df = pd.read_csv(METADATA_PATH)
print(f"Numri total i rreshtave në metadata: {len(df)}")

# 1. Mbushim vlerat që mungojnë për age me mesataren
mean_age = df['age'].mean()
df['age'] = df['age'].fillna(mean_age)

# 2. Kodojmë kolonat kategoriale (One-Hot Encoding)
df_encoded = pd.get_dummies(df, columns=['sex', 'localization'], drop_first=True)

# 3. Krijojmë etiketën target: nv -> 0 (nevi), mel -> 1 (melanoma)
df_encoded['label'] = df_encoded['dx'].map({'nv': 0, 'mel': 1})

print("Kolonat pas kodimit:")
print(df_encoded.columns.tolist())
df_encoded.head()

### 4. Ngarkimi i Imazheve dhe Eekstraktimi i Karakteristikave
Ndryshojmë përmasat e fotove në `32x32` piksela për të zvogëluar kompleksitetin e modelit, i kthejmë në vektorë 1D dhe i normalizojmë në intervalin `[0, 1]`.

In [ ]:
images = []
labels = []
meta_features = []

# Kolonat që duhen përjashtuar nga metadata për të marrë vetëm tiparet e pastra
exclude_cols = ['lesion_id', 'image_id', 'dx', 'dx_type', 'label']
meta_cols = [col for col in df_encoded.columns if col not in exclude_cols]

print("Duke ngarkuar fotot dhe duke nxjerrë tiparet...")
for idx, row in df_encoded.iterrows():
    img_id = row['image_id']
    label = row['label']
    
    # Kërko foton te mel ose nv
    img_path = os.path.join(MEL_DIR, f"{img_id}.jpg")
    if not os.path.exists(img_path):
        img_path = os.path.join(NV_DIR, f"{img_id}.jpg")
        
    if os.path.exists(img_path):
        with Image.open(img_path) as img:
            # Resize në 32x32 dhe konvertim në RGB
            img_resized = img.resize((32, 32)).convert('RGB')
            # Flatten dhe normalizim [0, 1]
            img_array = np.array(img_resized).flatten() / 255.0
            
            images.append(img_array)
            labels.append(label)
            meta_features.append(row[meta_cols].values.astype(float))
    else:
        print(f"Warning: Fotoja {img_id}.jpg nuk u gjet!")

X_img = np.array(images)
X_meta = np.array(meta_features)
y = np.array(labels)

print(f"Dimensioni i matricës së fotove: {X_img.shape}")
print(f"Dimensioni i matricës së metadata: {X_meta.shape}")
print(f"Dimensioni i target label y: {y.shape}")

### 5. Bashkimi i Karakteristikave (Feature Fusion) dhe Ndarja e Datasetit
Bashkojmë tiparet e fotove me metadata-n dhe ndajmë datasetin në **80% train** dhe **20% test** duke përdorur ndarje të stratifikuar bazuar në klasë.

In [ ]:
# Bashkojmë tiparet horizontally
X = np.hstack((X_img, X_meta))
print(f"Dimensioni total i matricës së inputit X: {X.shape}")

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Madhësia e Train Set: {X_train.shape[0]}")
print(f"Madhësia e Test Set: {X_test.shape[0]}")

### 6. Tuning i max_depth për të shmangur Overfitting
Pemët e vendimeve priren të bëjnë overfitting nëse nuk kufizojmë thellësinë e tyre (max_depth). Le të testojmë saktësinë për depth nga 1 deri në 14.

In [ ]:
depths = range(1, 15)
train_accs = []
test_accs = []

for d in depths:
    clf = DecisionTreeClassifier(max_depth=d, random_state=42)
    clf.fit(X_train, y_train)
    
    train_accs.append(accuracy_score(y_train, clf.predict(X_train)))
    test_accs.append(accuracy_score(y_test, clf.predict(X_test)))

# Vizualizimi
plt.figure(figsize=(10, 5))
plt.plot(depths, train_accs, label='Train Accuracy', marker='o', color='blue')
plt.plot(depths, test_accs, label='Test Accuracy', marker='o', color='green')
plt.xlabel('Thellësia Maksimale (max_depth)')
plt.ylabel('Saktësia (Accuracy)')
plt.title('Saktësia vs Max Depth')
plt.xticks(depths)
plt.legend()
plt.grid(True)
plt.show()

best_d = depths[np.argmax(test_accs)]
print(f"max_depth optimal është {best_d} me saktësi {max(test_accs):.4f} në test set.")

### 7. Trajnimi i Modelit Final
Duke përdorur depth optimal të gjetur më sipër (ose një vlerë të kontrolluar), trajnojmë modelin përfundimtar.

In [ ]:
optimal_depth = min(best_d, 6)
print(f"Duke trajnuar modelin e fundit me max_depth={optimal_depth}...")

model = DecisionTreeClassifier(max_depth=optimal_depth, random_state=42)
model.fit(X_train, y_train)

### 8. Vlerësimi i Modelit Final
Llogarisim saktësinë, printojmë raportin e klasifikimit (Precision, Recall, F1-Score) dhe vizualizojmë matricën e konfuzionit.

In [ ]:
y_pred = model.predict(X_test)

# Saktësia
print(f"Saktësia në Test Set: {accuracy_score(y_test, y_pred):.4f}\n")

# Classification Report
print("Raporti i Klasifikimit:")
print(classification_report(y_test, y_pred, target_names=['nv (Nevus)', 'mel (Melanoma)']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['nv', 'mel'])
disp.plot(cmap=plt.cm.Blues)
plt.title("Matrica e Konfuzionit")
plt.show()

### 9. Rëndësia e Karakteristikave (Feature Importance)
Le të analizojmë se sa ndikojnë tiparet e imazhit (pikselat) krahasuar me metadata-n.

In [ ]:
importances = model.feature_importances_
num_pixels = X_img.shape[1]

pixel_imp_sum = np.sum(importances[:num_pixels])
meta_imp = importances[num_pixels:]

print(f"Rëndësia e përgjithshme e pikselave të imazheve: {pixel_imp_sum:.4f}")
print("\nRëndësia e karakteristikave të metadata:")
for col, imp in zip(meta_cols, meta_imp):
    if imp > 0:
        print(f" - {col}: {imp:.4f}")

### 10. Vizualizimi i Pështatshëm i Pemës (Structure Visualization)
Vizualizojmë nivelet e para të Pemës së Vendimeve për të kuptuar si merren vendimet kryesore.

In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(
    model,
    max_depth=3,
    feature_names=[f"pixel_{i}" for i in range(num_pixels)] + meta_cols,
    class_names=['nv', 'mel'],
    filled=True,
    rounded=True,
    fontsize=10
)
plt.title("Vizualizimi i niveleve të para të Pemës së Vendimeve")
plt.show()

### 11. Ruajtja e Modelit të Trajnuar
Ruajmë modelin e trajnuar duke përdorur `joblib` në mënyrë që të mund ta ngarkojmë lehtësisht.

In [ ]:
model_path = os.path.join(BASE_DIR, "decision_tree_model.pkl")
joblib.dump(model, model_path)
print(f"Modeli u ruajt me sukses në: {model_path}")